In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.mnist import SimpleMNIST
from pt_to_api.utils import show_single_channel_red_green_black as S, to_show_list as tsl, zeros_with_1_at
import gc
import pickle
from pt_to_api.mnist import (
    SimpleMNIST,
    get_contribs_for_inp_vectorized,
    get_mnist_dataloader,
)
from pt_to_api.benchmark import train_x as TX
from pt_to_api import benchmark as B

In [ ]:
torch.set_printoptions(precision=4, sci_mode=False, linewidth=140)
np.set_printoptions(precision=4, linewidth=140)

In [ ]:
MODEL_PATH = Path("../../pt-to-api/data/model.pt")
INPUT_PATH = Path("../../pt-to-api/data/first-input-tens.pt")
MAIN_OUT_DIR = (Path.cwd() / "mnist-patches-data")
MAIN_OUT_DIR.mkdir(parents=True, exist_ok=True)
device = "cpu"

inp = torch.load(INPUT_PATH, weights_only=False)
model = SimpleMNIST()
model.load_state_dict(torch.load(MODEL_PATH))

dl = get_mnist_dataloader(0.2, bs=256, device=device)

In [ ]:
layer_key, channel = "layers.2", 12
input_act_key = "layers.1"
layer = model.get_submodule(layer_key)
patches_out_dir = MAIN_OUT_DIR / layer_key / str(channel)
patches_out_dir.mkdir(parents=True, exist_ok=True)


model = model.to(device)

In [ ]:
import torch.nn.functional as F
from torch import nn

# input_act = acts["layers.1"]

def get_indices_of_patches_to_extract(contribs, layer_name, channel, pos_threshes, neg_threshes):
    # print("slice", .shape, "thres", pos_threshes.shape, "neg", neg_threshes.shape)
    contrib_slice = contribs[layer_name][:, channel]
    pos_inds = torch.argwhere(contrib_slice >= pos_threshes)
    neg_inds = torch.argwhere(contrib_slice <= neg_threshes)
    return torch.cat([pos_inds, neg_inds])


def patches_of_single_batch_with_indices(input_act_of_batch, layer, indices):
    op_shape = get_output_shape(
        input_act_of_batch.shape, layer.kernel_size, layer.stride, layer.padding, layer.dilation
    )
    op_r, op_c = op_shape[-2], op_shape[-1]
    b = input_act_of_batch.shape[0]

    patches = F.unfold(
        input_act_of_batch, layer.kernel_size, layer.dilation, layer.padding, layer.stride
    ).reshape(b, -1, op_r, op_c)

    return torch.stack([
        patches[ind[0], :, ind[1], ind[2]]
        for ind in indices
    ])


def get_output_shape(input_shape, ksize, stride, padding, dilation):
    B, C, H, W = input_shape
    Ho = (H + 2*padding[0] - dilation[0]*(ksize[0]-1) - 1) // stride[0] + 1
    Wo = (W + 2*padding[1] - dilation[1]*(ksize[1]-1) - 1) // stride[1] + 1
    return (B, C, Ho, Wo)


In [ ]:
def get_contribs_for_batch(batch, targets, device):
    targs = torch.concat([zeros_with_1_at(10, targ) for targ in targets]).to(device)
    contribs, acts, params = get_contribs_for_inp_vectorized(
        batch, model, targs, "layers.5", device
    )
    return contribs, acts, params

In [ ]:
batch, targets = next(iter(dl.train))

In [ ]:
contribs, acts, params = get_contribs_for_batch(batch, targets, device)

In [ ]:
# need for torch load (in colab these were defined in the notebook)
Autoencoder = TX.Autoencoder
SingleRun = TX.SingleRun

In [ ]:
SHAPE = (8,9)

In [ ]:
layer_name = "layers.2"
channel = 12
d12 = Path.home() / "Desktop/gdrive-sync/l2-o12-patches/cosine-anneal/comps/12"
pos_threshes = torch.load(d12 / "pos_threshes.pt", weights_only=True)
neg_threshes = torch.load(d12 / "neg_threshes.pt", weights_only=True)
run_path = next((d12 / "pt").glob("*.pt"))
run = torch.load(run_path, weights_only=False)
run.model.eval()
layer = model.get_submodule(f"{layer_name}")
weight = layer.weight[channel].clone().detach()
weight = weight.reshape(-1).numpy()

In [ ]:
zeroth_act_inds

In [ ]:
indices = get_indices_of_patches_to_extract(contribs, layer_key, channel, pos_threshes, neg_threshes)
# zeroth_act_inds = indices[indices[:,0] == 0]
patches_of_batch = patches_of_single_batch_with_indices(acts[input_act_key], layer, indices)
patches_of_batch = patches_of_batch.numpy()

In [ ]:
# this scaled_pw is from colab
# ive got a different scaled_pw in downstream training, which is most likely not very nice
training_scaled_pw = torch.load("/Users/hariomnarang/Desktop/gdrive-sync/l2-o12-patches/cosine-anneal/scaled_pw.pt", weights_only=False)
pw = patches_of_batch * weight
scaler = B.NormaliseStdScaler().fit(training_scaled_pw)
scaled_pw = scaler.transform(pw)

In [ ]:
del training_scaled_pw
gc.collect()

In [ ]:
sd = torch.load("./l2-o12-main-saved-model.pt")
model = TX.Autoencoder(72, 6)
model.load_state_dict(sd)
model.eval()

In [ ]:
with torch.no_grad():
    recon, codes, _ = model(torch.tensor(scaled_pw, dtype=torch.float32))

In [ ]:
# not all that bad
idxs = torch.randperm(recon.shape[0])[:15]

for i in idxs:
    print(i)
    show_72_list([recon[i], scaled_pw[i]])
    plt.show()

In [ ]:
from pt_to_api.utils import show_72_list

show_72_list(tsl(scaled_pw))
show_72_list(tsl(scaled_pw / weight))

In [ ]:
from pt_to_api.utils import otsu_threshold
otsu_threshold(codes[:,5])

In [ ]:
# for now, its okay, we dont have a distribution where 
# code is negative high and positive high simultaneously
# in that case, the pattern is opposite though, so we would want to
# consider them separate (if we have to thresholds, positive and negative, we need to keep them separate.)
# for now, manual step again
# there is a lot of manual steps from my side.
# need to write them down
# and cleanup the notebook
def is_active(code, threshold):
    if threshold < 0:
        return code <= threshold
    else:
        return code >= threshold

In [ ]:
# now the hard part, we want to know if something is activated. hmmmmmmmm
# tis gona be complicatessssss for more data. i can otsu it right now though, lets try that
thresholds = [otsu_threshold(codes[:, c]) for c in range(codes.shape[1])]
# for c in range(codes.shape[1]):
#     print(c)
#     plt.hist(codes[:, c])
#     plt.show()

In [ ]:
# now, go through every patches, find activated components
# keep track of the index of the patch and activated component
# see if they "match"
from collections import defaultdict

comp_idx_by_sample_idxs = defaultdict(list)

for sample_idx in range(len(codes)):
    _codes = codes[sample_idx]
    actives = []
    for comp_idx, (c,t) in enumerate(zip(_codes, thresholds)):
        if is_active(c, t):
            actives.append(comp_idx)
    comp_idx_by_sample_idxs[tuple(actives)].append(sample_idx)

In [ ]:
for k,v in comp_idx_by_sample_idxs.items():
    print(k, len(v))

In [ ]:
acts[input_act_key].shape, scaled_pw.shape

In [ ]:
scaled_pw[0].shape

In [ ]:
# lets look at some samples with 0th code only active
import random
from pt_to_api.utils import get_receptive, mk_rect_on_ax
# these are actually now, the "combinations", or dict learning components

sample_idxs_for_0 = comp_idx_by_sample_idxs[(1,4,5)]

idxs = random.sample(sample_idxs_for_0, 3)

for idx in idxs:
    # now what,mmmmmmm

    slice_num, y, x = indices[idx]
    (y0,x0), (y1,x1) = get_receptive(y, x, layer.kernel_size, layer.stride, layer.padding, layer.dilation)

    axes = S([a for a in acts[input_act_key][slice_num]], (20,5), 8)
    for ax in axes:
        mk_rect_on_ax(ax, y0, x0, y1-y0, x1-x0)
    plt.show()
    show_72(scaled_pw[idx])
    plt.show()


In [ ]:
# lets look at some samples with 0th code only active
import random
from pt_to_api.utils import get_receptive, mk_rect_on_ax
# these are actually now, the "combinations", or dict learning components

sample_idxs_for_0 = comp_idx_by_sample_idxs[(0,1)]

idxs = random.sample(sample_idxs_for_0, 6)

for idx in idxs:
    # now what,mmmmmmm

    slice_num, y, x = indices[idx]
    (y0,x0), (y1,x1) = get_receptive(y, x, layer.kernel_size, layer.stride, layer.padding, layer.dilation)

    axes = S([a for a in acts[input_act_key][slice_num]], (20,5), 8)
    for ax in axes:
        mk_rect_on_ax(ax, y0, x0, y1-y0, x1-x0)
    plt.show()
    show_72(scaled_pw[idx])
    plt.show()


In [ ]:
# lets look at some samples with 0th code only active
import random
from pt_to_api.utils import get_receptive, mk_rect_on_ax
# these are actually now, the "combinations", or dict learning components

sample_idxs_for_0 = comp_idx_by_sample_idxs[(1,5)]

idxs = random.sample(sample_idxs_for_0, 6)

for idx in idxs:
    # now what,mmmmmmm

    slice_num, y, x = indices[idx]
    (y0,x0), (y1,x1) = get_receptive(y, x, layer.kernel_size, layer.stride, layer.padding, layer.dilation)

    axes = S([a for a in acts[input_act_key][slice_num]], (20,5), 8)
    for ax in axes:
        mk_rect_on_ax(ax, y0, x0, y1-y0, x1-x0)
    plt.show()
    show_72(scaled_pw[idx])
    plt.show()


In [ ]:
comps = model.decoder.weight.T.clone().detach().numpy()
for c in comps:
    show_72(c)
# S([c.reshape(SHAPE) for c in comps], (20,5), len(comps))
# S([c.reshape(SHAPE) for c in run.components], (20,5), len(run.components) // 2)
plt.show()

In [ ]:
# what do i do now? see how each component looks on the input activation
# each position basically, to see what each component "means"
# for this, we need to see the input activation fully
# for each active component, draw rectangles

In [ ]:
# these are the input activations


In [ ]:
zeroth_act_inds

In [ ]:
# we have 72 indices, which is 8*3*3 (a single patch)
# we have the index of the patch, is the index related to input or output?
# this is the index of the olutput, 0th batch slice, 4th output row, 5th output column
# we need the receptive field
from pt_to_api.utils import get_receptive, mk_rect_on_ax

slice_num, y, x = zeroth_act_inds[0]
(y0,x0), (y1,x1) = get_receptive(y, x, layer.kernel_size, layer.stride, layer.padding, layer.dilation)

In [ ]:
one_input_act = acts[input_act_key][slice_num]

one_input_act[:, y0:y1, x0:x1]

In [ ]:
axes = S([a for a in acts[input_act_key][slice_num]], (20,5), 8)
for ax in axes:
    mk_rect_on_ax(ax, y0, x0, y1-y0, x1-x0)

plt.show()

In [ ]:
from pt_to_api.utils import show_72

codes[0]
active_codes = [0]

show_72(scaled_pw[0])
show_72(comps[0])


In [ ]:
codes[1]
S([(codes[1][j] * comps[j]).reshape(SHAPE) for j in range(len(codes[1]))], (20,5), len(codes[1]), ax_titles=[f"{c:.4f}" for c in codes[1]])
plt.show()

In [ ]:
active_codes = [1, 4, 5]

show_72(weight)
show_72(scaled_pw[1] / weight)
show_72(scaled_pw[1])
plt.show()
for c in active_codes:
    show_72(comps[c])
    plt.show()

In [ ]:
slice_num, y, x = zeroth_act_inds[1]
(y0,x0), (y1,x1) = get_receptive(y, x, layer.kernel_size, layer.stride, layer.padding, layer.dilation)
axes = S([a for a in acts[input_act_key][slice_num]], (20,5), 8)
for ax in axes:
    mk_rect_on_ax(ax, y0, x0, y1-y0, x1-x0)

plt.show()

We need better fitting lol. But, lets still continue a bit.  

In [ ]:
?mk_rect_on_ax

In [ ]:
layer_name = "layers.2"
channel = 12
d12 = Path.home() / "Desktop/gdrive-sync/l2-o12-patches/cosine-anneal/comps/12"
pos_threshes = torch.load(d12 / "pos_threshes.pt", weights_only=True)
neg_threshes = torch.load(d12 / "neg_threshes.pt", weights_only=True)
run_path = next((d12 / "pt").glob("*.pt"))
run = torch.load(run_path, weights_only=False)
run.model.eval()
layer = model.get_submodule(f"{layer_name}")
weight = layer.weight[channel].clone().detach()
weight = weight.reshape(-1).numpy()

In [ ]:
# a problem is that the codes dont look very normalized
# how do i fix that?
# i could normalize the codes after a loop, and try to update the weights?
# hmmmm
with np.printoptions(suppress=True, precision=5):
    print(codes[0].numpy())

We have another problem now, codes are not scaled at all.  
I can't make correct deductions using this.  


How do i scale the codes? hmmmmmmmmmm

There are other things also, technically, the final inversed value is more useful, then just codes. why?  
Cuz the final inverse value is the truth. values stored in components would have a 0 where there might be a high value (cuz we take mean).   

It might be best to inverse the components and inverse the result of scaled components.  
What after this?  


aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa ;_;   
tis sucks.  
hmpf.  


mmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmm aaaaaaaaaaaaaaaaaaaaaaaaaaaaa.  

In [ ]:
(codes[0] / codes[0].abs().max() )

In [ ]:
# isme to sabmei kuch aara hai bc hmpfffffff

# and some are very weaks also. aaaaa
# it might be cuz of mean shifting.
comps = []
inv_comps = []
for j in range(len(codes[0])):
    comp = codes.numpy()[0][j] * run.components[j]
    comps.append(comp.reshape(SHAPE))
    inv_comps.append(scaler.inverse_transform(comp).reshape(SHAPE))

S(comps, (20,5), len(comps)//2)
plt.show()

In [ ]:
S(inv_comps, (20,5), len(inv_comps)//2)
plt.show()

In [ ]:
S([np.array(comps).sum(axis=0), recon[0].reshape(SHAPE)])

In [ ]:

S([scaler.inverse_transform(
    np.array(comps).sum(axis=0).reshape(-1)
).reshape(SHAPE), pw[0].reshape(SHAPE)], viztype="global")

In [ ]:
i = 0
S([
    pw[i].reshape(SHAPE), 
    scaler.inverse_transform(recon[i]).reshape(SHAPE), 
    np.array(comps).sum(axis=0)], (10,3), ncols=3, viztype="local")
plt.show()

In [ ]:
# scaled_pw = scaled_pw[sampled_idxs]

In [ ]:
S([recon.reshape(SHAPE)])

In [ ]:
# what does this mean?
zeroth_act_inds

In [ ]:
patches_of_batch.shape